# XBTorch::Example 08:Fault Tolerance Algorithms

## Introduction

In this example, we will explore how fault-tolerance algorithms can be implemented with XBTorch's inference accelerator worklflow.  We will use the same running example of the two-layer perceptron network pre-trained using XBTorch. In particular, we will compare the performance of 3 fault-tolerance algorithms in improving neural network performance with inference accelerator noise sources present.

- Layer ensemble averaging (LEA) [1]
- Mapping Algorithm with inner-fault TOlerance (MAO) [2]
- Committee Machines [3]

## Getting Started

Let's import necessary dependencies.

In [1]:
# General imports
import numpy as np
import matplotlib.pyplot as plt
import random
import time
from pathlib import Path
import pickle

import torch
import torch.nn as nn

from torchvision import datasets, transforms
from torch.utils.data import DataLoader, ConcatDataset

from functools import partial
import torch.optim as optim

In [2]:
# XBTorch imports
import xbtorch

from xbtorch.devices import AnalyticalIdeal, AnalyticalReal, TabularAnalyticalReal, TabularCompactFeFETKriging, TabularExperimentalFemFETKriging
from xbtorch.patches import xbtorch_model

from xbtorch.nn.utils import test_classifier

from xbtorch.deployment import SimpleFixedPoint, map_random, encode_simple_binary, encode_MAO, encode_LEA1, encode_LEA2, test_committee

from nets.mlp import SimpleMLP

W0930 21:38:44.163000 2149652 torch/utils/cpp_extension.py:2425] TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
W0930 21:38:44.163000 2149652 torch/utils/cpp_extension.py:2425] If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'] to specific architectures.


In [3]:
seed = 0

torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed) # controls weight updatejump table stochasticity

# Check if CUDA is available and select the device
device = torch.device("cuda:5" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda:5


## XBTorch Parameters

In [4]:
xb_size = (2500, 2500)
g_min = 133
g_max = 233
read_noise = 10
write_noise = 50
weight_encoding_scheme = encode_simple_binary
mapping_scheme = map_random
output_polling_mode = "avg"
adc_precision = dac_precision = 8

wage_params = { "wl_weight": 2, # 2 = ternary weights
                "wl_grad": 8,
                "wl_activation": 8,
                "wl_error": 8,
                "rounding_weight" : "nearest",
                "rounding_activation" : "nearest",
                "rounding_grad" : "nearest",
                "rounding_error" : "nearest",
               }

## Prepare the dataset

We can now prepare the dataset for our neural network.

In [5]:
# Define transforms to apply to the data
transform = transforms.Compose([
    transforms.ToTensor(),  # Convert images to tensors
    transforms.Normalize((0.1307,), (0.3081,))  # Normalize the image data
])

# Load the MNIST training and test datasets
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# Create data loaders for batching and shuffling
train_loader = DataLoader(train_dataset, batch_size=4096, shuffle=True, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=10000, shuffle=False, num_workers=4)

## Baseline network

We can now benchmark the performance of our network. Let's first start without any fault-tolerance algorithm in place, so as to establish a baseline for performance comparisons that we'll do later.

In order to gauge the performance of fault-tolerance algorithms, it will be helpful to study increasing levels of faults. For this purpose, we will now introduce (and sweep over) stuck devices to our inference accelerator. 

XBTorch provides two primary parameters for controlling the nature of stuck-at faults in the simulated accelerator.


| Parameter    | Purpose |
| -------- | ------- |
| `stuck_percentage`  | Out of the `xb_size` devices, how many should be stuck (either high or low with equal probability. The sampling is done uniformly at random     |
| `stuck_mode` | Dictates the conductance that stuck-at devices end up taking. Options: `ideal` or `real` |

When the ideal stuck mode is used, stuck-at low and stuck-at high devices take conductances $G_{min}$ and $G_{max}$ respectively. When the real stuck mode is used, stuck-at low and stuck-at high devices take conductances $G_{stuck, low}$ and $G_{stuck, high}$ respectively (default values are 10 $\mu S$ and 500 $\mu S$). The real mode enables investigations where stuck devices can have conductances significantly outside the range of non-stuck i.e. operable devices. All other parameters remain similar as Example 07 (HWA Inference).

For more details on the API, view the full XBTorch documentation.

In [6]:
# Model parameters
input_size = 28 * 28
hidden_size = 150
output_size = 10

# Stuck-at Fault parameters
stuck_mode = "real"
iterations = 10 # due to randomness, better to record data over multiple iterations 
stuck_percentages = [0, 0.1, 0.2] # Sweep over stuck percentage

In [7]:
baseline_accs = np.zeros((1, len(stuck_percentages), iterations, 1))

for j, stuck_percentage in enumerate(stuck_percentages):
    for i in range(iterations):
        # Initialize the inference accelerator
        inference_accelerator = SimpleFixedPoint(g_min=g_min, 
                                                 g_max=g_max, 
                                                 adc_bits=adc_precision, # use sweep value
                                                 dac_bits=dac_precision, # use sweep value
                                                 read_noise=read_noise, # no read noise
                                                 write_noise=write_noise, # no write noise 
                                                 xb_size=xb_size,
                                                 weight_encoding_scheme=weight_encoding_scheme, 
                                                 xb_mapping_scheme=mapping_scheme,
                                                 stuck_percentage=stuck_percentage,
                                                 stuck_mode=stuck_mode,
                                                 device=device)
        # Initialize XBTorch
        xbtorch.initialize(pytorch_device=device,
                       inference_accelerator=inference_accelerator,
                       wage_quantize=True, # since the trained solution utilized wage quantization
                       wage_params=wage_params
                       )
    
        # Initialize the chip and the mappings
        inference_accelerator.initialize_chip()
    
        # Create, patch, and load the model's state dictionary (same as above)
        model = SimpleMLP(input_size, hidden_size, output_size).to(device)
        model = xbtorch_model(model)
        
        with open("checkpoints/hwa_train_mlp_hwa_example.pkl", "rb") as f:
            full_weights_hwa = pickle.load(f)
            
        epoch = -1 
        
        for name, param in model.named_parameters():
            new_tensor = torch.from_numpy(full_weights_hwa[name][epoch, ...]).to(
                device=param.device,
                dtype=param.dtype,
            )
            param.data = new_tensor
    
        # Initialize array mappings
        model.initialize_array_mappings(output_polling_mode=output_polling_mode, 
                                        additional_args={}, 
                                        existing_mappings=[]
                                       )
    
        # Test the network's performance
        model.xb_eval(enable=True)
        acc, _ = test_classifier(test_loader, model, device)
        baseline_accs[0, j, i] = acc

## Layer Ensemble Averaging

In LEA, each encoded conductance matrix must be mapped redundantly to the inference accelerator. During inference, output currents have to be averaged for each layer before being sent to following layers. For a particular layer, the averaging isn't necessarily over all output currents. The process can discard outputs as needed (for example, one may want to discard outputs from a severely defective row).

LEA has 2 primary hyperparameters, $\alpha$ and $\beta$. XBTorch follows the following convention for these:

- Each layer is mapped redundantly $\beta$ times.
- Out of the $\beta$ mappings, a total of $\alpha$ mappings participate in the averaging process for each output.

For this example demonstration, since device defects are uniformly distributed, we will set $\alpha = \beta$. This makes our analysis simpler and avoids unnecessary work. If the defects were not uniformly distributed, then dropping out outputs from severely defective rows would be a plausible consideration.

In [8]:
alphas_betas = [1, 3, 6]

We need to tell XBTorch how to gather currents for a layer during inference. The pre-implemented `"reduced_avg"` mode implements the reduced averaging operation required by LEA. See the implementation of `xbtorch_layer::xbtorch_forward()` in `xbtorch.patches.decorators` for more details.

In [9]:
output_polling_mode = "reduced_avg"

We also need to specify a weight encoding scheme. This is what dictates how weights are converted to device conductances. See `xbtorch.deployment.encoding` for more details.

In [10]:
weight_encoding_scheme = encode_LEA1

Finally, we need to specify where encoded weights would be mapped on the inference accelerator. Again, since defects  are distributed randomly, we simply use the `map_random` function to avoid unnecessary work searching the entire accelerator for portions with a lower degree of defects. Since producing the mapping information requires prior knowledge on the redundancy, we do this inside the full sweep loop.

In [11]:
lea_accs = np.zeros((len(alphas_betas), len(stuck_percentages), iterations, 1))

for k, alpha_beta in enumerate(alphas_betas):
    mapping_scheme = partial(map_random, beta=alpha_beta)
    additional_args = {"alpha": alpha_beta, "beta": alpha_beta}
    for j, stuck_percentage in enumerate(stuck_percentages):
        for i in range(iterations):
            # Initialize the inference accelerator
            inference_accelerator = SimpleFixedPoint(g_min=g_min, 
                                                     g_max=g_max, 
                                                     adc_bits=adc_precision, # use sweep value
                                                     dac_bits=dac_precision, # use sweep value
                                                     read_noise=read_noise, # no read noise
                                                     write_noise=write_noise, # no write noise 
                                                     xb_size=xb_size,
                                                     weight_encoding_scheme=weight_encoding_scheme, 
                                                     xb_mapping_scheme=mapping_scheme,
                                                     stuck_percentage=stuck_percentage,
                                                     stuck_mode=stuck_mode,
                                                     device=device)
            # Initialize XBTorch
            xbtorch.initialize(pytorch_device=device,
                           inference_accelerator=inference_accelerator,
                           wage_quantize=True, # since the trained solution utilized wage quantization
                           wage_params=wage_params
                           )
        
            # Initialize the chip and the mappings
            inference_accelerator.initialize_chip()
        
            # Create, patch, and load the model's state dictionary (same as above)
            model = SimpleMLP(input_size, hidden_size, output_size).to(device)
            model = xbtorch_model(model)
            
            with open("checkpoints/hwa_train_mlp_hwa_example.pkl", "rb") as f:
                full_weights_hwa = pickle.load(f)
                
            epoch = -1 
            
            for name, param in model.named_parameters():
                new_tensor = torch.from_numpy(full_weights_hwa[name][epoch, ...]).to(
                    device=param.device,
                    dtype=param.dtype,
                )
                param.data = new_tensor
        
            # Initialize array mappings
            model.initialize_array_mappings(output_polling_mode=output_polling_mode, 
                                            additional_args=additional_args, 
                                            existing_mappings=[]
                                           )
        
            # Test the network's performance
            model.xb_eval(enable=True)
            acc, _ = test_classifier(test_loader, model, device)
            lea_accs[k, j, i] = acc

## Mapping Algorithm with inner-fault tOlerance 

In MAO, each encoded conductance matrix must be mapped redundantly to the inference accelerator. During inference, output currents have to be summed for each layer before being sent to following layers. For a particular layer, the summing is necessarily over all output currents.

As we did for LEA, we can set the polling mode and the encoding scheme appropriately.

In [12]:
weight_encoding_scheme = encode_MAO
output_polling_mode = "sum"

Everything else remains similar as before. Let's perform the sweep!

In [ ]:
mao_accs = np.zeros((len(alphas_betas), len(stuck_percentages), iterations, 1))

for k, alpha_beta in enumerate(alphas_betas):
    mapping_scheme = partial(map_random, beta=alpha_beta)
    additional_args = {"alpha": alpha_beta, "beta": alpha_beta}
    for j, stuck_percentage in enumerate(stuck_percentages):
        for i in range(iterations):
            # Initialize the inference accelerator
            inference_accelerator = SimpleFixedPoint(g_min=g_min, 
                                                     g_max=g_max, 
                                                     adc_bits=adc_precision, # use sweep value
                                                     dac_bits=dac_precision, # use sweep value
                                                     read_noise=read_noise, # no read noise
                                                     write_noise=write_noise, # no write noise 
                                                     xb_size=xb_size,
                                                     weight_encoding_scheme=weight_encoding_scheme, 
                                                     xb_mapping_scheme=mapping_scheme,
                                                     stuck_percentage=stuck_percentage,
                                                     stuck_mode=stuck_mode,
                                                     device=device)
            # Initialize XBTorch
            xbtorch.initialize(pytorch_device=device,
                           inference_accelerator=inference_accelerator,
                           wage_quantize=True, # since the trained solution utilized wage quantization
                           wage_params=wage_params
                           )
        
            # Initialize the chip and the mappings
            inference_accelerator.initialize_chip()
        
            # Create, patch, and load the model's state dictionary (same as above)
            model = SimpleMLP(input_size, hidden_size, output_size).to(device)
            model = xbtorch_model(model)
            
            with open("checkpoints/hwa_train_mlp_hwa_example.pkl", "rb") as f:
                full_weights_hwa = pickle.load(f)
                
            epoch = -1 
            
            for name, param in model.named_parameters():
                new_tensor = torch.from_numpy(full_weights_hwa[name][epoch, ...]).to(
                    device=param.device,
                    dtype=param.dtype,
                )
                param.data = new_tensor
        
            # Initialize array mappings
            model.initialize_array_mappings(output_polling_mode=output_polling_mode, 
                                            additional_args=additional_args, 
                                            existing_mappings=[]
                                           )
        
            # Test the network's performance
            model.xb_eval(enable=True)
            acc, _ = test_classifier(test_loader, model, device)
            mao_accs[k, j, i] = acc

## Committee Machines

In LEA and MAO, the same solution is mapped redundantly to the inference accelerator. In CM, this changes, and instead we map multiple pre-trained solutions to the inference accelerator. During inference, there is no averaging at the layer level, meaning each individual solution is mapped a single time.

Below is our recommended way of implementing CM fault-tolerance using XBTorch. Individual models to be mapped are gathered and written to the same inference accelerator at non-conflicting locations. The output polling mode isn't important here, since summation and averaging are equivalent since each individual network layer is its own entity/standalone (unlike LEA or MAO). What is important, however, is that outputs of the final layer are averaged before making any predictions. XBTorch provides a helper function, `test_committee()`, for this purpose. Refer to `xbtorch.deployment.committee` for the full implementation. 

Since we haven't explicitly trained multiple HWA networks in our running 2-layer perceptron example, we will use the same solution which we already have from the previous example notebooks on HWA training. In a practical implementation, one should use distinct fully converged solutions as committee members.

In [ ]:
encoding_scheme = encode_simple_binary
mapping_scheme = partial(map_random, beta=1)
output_polling_mode = "avg" # could've been sum as well, identical since \beta = 1'

In [ ]:
cm_accs = np.zeros((len(alphas_betas), len(stuck_percentages), iterations, 1))

for k, alpha_beta in enumerate(alphas_betas):

    for j, stuck_percentage in enumerate(stuck_percentages):
        # Initialize the inference accelerator
        inference_accelerator = SimpleFixedPoint(g_min=g_min, 
                                                 g_max=g_max, 
                                                 adc_bits=adc_precision, # use sweep value
                                                 dac_bits=dac_precision, # use sweep value
                                                 read_noise=read_noise, # no read noise
                                                 write_noise=write_noise, # no write noise 
                                                 xb_size=xb_size,
                                                 weight_encoding_scheme=weight_encoding_scheme, 
                                                 xb_mapping_scheme=mapping_scheme,
                                                 stuck_percentage=stuck_percentage,
                                                 stuck_mode=stuck_mode,
                                                 device=device)
        # Initialize XBTorch
        xbtorch.initialize(pytorch_device=device,
                       inference_accelerator=inference_accelerator,
                       wage_quantize=True, # since the trained solution utilized wage quantization
                       wage_params=wage_params
                       )
    
        # Initialize the chip and the mappings
        inference_accelerator.initialize_chip()

        committee_models = []
        existing_mappings = [] # providing this (used as a reference internally) ensures that subsequent models do not overlap with existing model mappings on the xbar
        
        # Gather the models for the committee (create, load state dicts, patch for XBTorch)
        for l in range(alpha_beta):
            # Create, patch, and load the model's state dictionary
            model = SimpleMLP(input_size, hidden_size, output_size).to(device)
            model = xbtorch_model(model)
            
            with open("checkpoints/hwa_train_mlp_hwa_example.pkl", "rb") as f:
                full_weights_hwa = pickle.load(f)
                
            epoch = -1
    
            for name, param in model.named_parameters():
                new_tensor = torch.from_numpy(full_weights_hwa[name][epoch, ...]).to(
                    device=param.device,
                    dtype=param.dtype,
                )
                param.data = new_tensor
    
            # Initialize array mappings and enable HWA evaluation
            model.initialize_array_mappings(output_polling_mode=output_polling_mode, 
                                            additional_args={}, 
                                            existing_mappings=existing_mappings
                                           )
            model.xb_eval(enable=True)
    
        committee_models.append(model)

        # inference_accelerator.plot_array() # uncomment to check how the committee looks on the accelerator!
        for i in range(iterations):
            acc = test_committee(test_loader, committee_models, device)
            cm_accs[k, j, i] = acc

In [ ]:
def plot_accuracy_bars(data, redundancy_values, stuck_percentages, title):
    """
    Plots bar charts of neural network accuracies.
    
    Parameters:
    - data: numpy array with shape (len(redundancy_values), len(stuck_percentages), iterations, 1)
    - redundancy_values: list of redundancy values (e.g., [1, 3, 6])
    - stuck_percentages: list of stuck percentages (e.g., [0, 0.1, 0.2])
    """
    # data = np.squeeze(data)  # remove singleton dimensions
    avg_data = data.mean(axis=-2)  # average across iterations, shape (redundancy, stuck)
    avg_data = np.squeeze(avg_data, axis=-1)
    
    num_stuck = len(stuck_percentages)
    x = np.arange(len(redundancy_values))  # label locations

    bar_width = 0.1
    fig, ax = plt.subplots()

    for i, stuck in enumerate(stuck_percentages):
        # Bar positions with slight offsets
        offset = (i - num_stuck/2) * bar_width + bar_width/2
        bar_positions = x + offset
        heights = avg_data[:, i]
        ax.bar(bar_positions, heights, width=bar_width, label=f"Stuck {round(stuck * 100)} %")

    ax.set_xlabel("Redundancy")
    ax.set_ylabel("Accuracy (%)")
    ax.set_xticks(x)
    ax.set_xticklabels(redundancy_values)
    ax.legend()
    plt.title(title)

    plt.show()

In [ ]:
# Save statistics for plotting later, if necessary (can be skipped)
fault_tolerance_dict = {"baseline_accs": baseline_accs,
                        "lea_accs": lea_accs,
                        "mao_accs": mao_accs,
                        "cm_accs": cm_accs,
                        "alphas_betas": alphas_betas,
                        "stuck_percentages": stuck_percentages}

output_path = "checkpoints/"
Path(output_path).mkdir(parents=False, exist_ok=True)
model_name = f"{output_path}/fault_tolerance_example.pkl"
with open(model_name, "wb") as f:
    pickle.dump(fault_tolerance_dict, f)

In [ ]:
# Load statistics
with open("checkpoints/fault_tolerance_example.pkl", "rb") as f:
    fault_tolerance_dict = pickle.load(f)

In [ ]:
plot_accuracy_bars(fault_tolerance_dict["baseline_accs"], [1], fault_tolerance_dict["stuck_percentages"], title="Baseline")

In [ ]:
plot_accuracy_bars(fault_tolerance_dict["lea_accs"], fault_tolerance_dict["alphas_betas"], fault_tolerance_dict["stuck_percentages"], title="LEA")

In [ ]:
plot_accuracy_bars(fault_tolerance_dict["mao_accs"], fault_tolerance_dict["alphas_betas"], fault_tolerance_dict["stuck_percentages"], title="MAO")

In [ ]:
plot_accuracy_bars(fault_tolerance_dict["cm_accs"], fault_tolerance_dict["alphas_betas"], fault_tolerance_dict["stuck_percentages"], title="CM")

## Conclusion

We explored how advanced fault-tolerance schemes could be implemented and compared using XBTorch's hardware-aware inference feature. We were able to specify characteristics of stuck-at fault devices (percentage, mode, etc.), and gauge how different fault-tolerance schemes behave in recovering network performance under these faults. Results highlight that layer ensemble averaging is the most successful in tolerating the kinds of stuck-at faults that we have injected in this notebook. This is in line with results reported in literature [1]. 

## References

1. Yousuf, O., Hoskins, B. D., Ramu, K., Fream, M., Borders, W. A., Madhavan, A., ... & Adam, G. C. (2025). Layer ensemble averaging for fault tolerance in memristive neural networks. Nature Communications, 16(1), 1250.
2. Huangfu, W., Xia, L., Cheng, M., Yin, X., Tang, T., Li, B., ... & Yang, H. (2017, January). Computation-oriented fault-tolerance schemes for RRAM computing systems. In 2017 22nd Asia and South Pacific Design Automation Conference (ASP-DAC) (pp. 794-799). IEEE.
3. Joksas, D., Freitas, P., Chai, Z., Ng, W. H., Buckwell, M., Li, C., ... & Mehonic, A. (2020). Committee machines—a universal method to deal with non-idealities in memristor-based neural networks. Nature communications, 11(1), 4273.